In [ ]:
import pandas as pd

# Path to metadata
df_path = "D:\DATA\with_snomed_category.csv"
df_all = pd.read_csv(df_path)

print(df_all.columns)

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
from helper_functions import subset_df, subset_df_list
df_HE = df_all.copy()
#df_HE = subset_df(df_all, "stain", "HE")
#df_HE = subset_df(df_HE, "mattype tekst", "Hist. store")
#df_HE = subset_df_list(df_HE, "T_category", "Placenta, Fetal Membranes, and Fetus")

In [ ]:
# Feature Results
feature_result = r"D:\NOTEBOOKS\Christine\all_slides\combined_feature_summary.csv"
df_feature_result = pd.read_csv(feature_result)

df_feature_result = df_feature_result[df_feature_result['status'] == "feature extraction complete"].copy()
df_uni = df_feature_result[df_feature_result['model'] == "uni"].copy()
uni = set(df_uni["wsi_path"])

df_conch = df_feature_result[df_feature_result['model'] == "conch"].copy()
conch = set(df_conch["wsi_path"])

df_hopt = df_feature_result[df_feature_result['model'] == "h-optimus-0"].copy()
hopt = set(df_hopt["wsi_path"])

with_features = uni & conch & hopt
print("WSIs with features across uni, conch & h-optimus-0: ", len(with_features))

In [ ]:
df_HE = df_HE[df_HE["filename"].isin(with_features)]
files = df_HE["filename"].tolist()

print("Number of files: ", len(files))

In [ ]:
df_HE["T_category"].value_counts().head(10)

In [ ]:
from visualize_features2 import FeatureDataBuilder

zarr_dir = r"Q:\NOTEBOOKS\Christine\all_slides\zarr"
cache_file = "cache_all_slides.pkl"
models = ["conch", "uni", "h-optimus-0"]

featurebuilder = FeatureDataBuilder(files, df_all, zarr_dir, cache_file, models)

In [ ]:
df = featurebuilder.df_merged

In [ ]:
from visualize_features import FeatureVisualizer

categories = ["T_category", "T_text", "M_category", "M_text", "team", "sex", "alder", "alder gruppe", 'mattype tekst', 'stain', 'snomed_code',
              'snomed_text', 'undersoeger_anonymous'] 

for model in models:
    feat_col = f"features_{model}"
    for cat in categories:
        try:
            print(f"\n Visualizing model: {model}, feature: {cat}")
            viz = FeatureVisualizer(df, label_col = cat, features_col = feat_col, artifact_col = "artifact_default_pct")
            viz.print_score()
            viz.tsne_plot()
            
        except Exception as e:
            print(f"Error visualizing projection: {e}")
            continue